In [ ]:
!pip install biopython pandas numpy scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np
from Bio import SeqIO
import random

Paths

In [ ]:
TSS_PATH = "x"
GENOME_PATH = "x"
GENE_PATH = "x"

TSS

In [ ]:
tss_df = pd.read_csv(TSS_PATH, sep="\t")
tss_df.head()


tss_df = tss_df.dropna(subset=["pos_1", "strand"]).copy()
tss_df["pos_1"] = tss_df["pos_1"].astype(int)

# keeping just + and -
tss_df = tss_df[tss_df["strand"].isin(["+", "-"])].copy()

print("Number of TSS rows:", len(tss_df))
tss_df.head()

Number of TSS rows: 2122


,start,end,pos_1,strand,genes,promoter
0,114,114,114,+,thrA; thrL,NaN
1,147,147,147,+,thrA; thrL,NaN
2,8015,8015,8015,-,yaaJ; talB,NaN
3,8192,8192,8192,+,yaaJ; talB,NaN
4,10564,10564,10564,-,mbiA; satP; yaaW,NaN


Genome

In [ ]:
genome_record = next(SeqIO.parse(GENOME_PATH, "fasta"))
genome_seq = str(genome_record.seq).upper()
genome_len = len(genome_seq)

print("Genome length:", genome_len)
print(tss_df.head())

Genome length: 4641652
   start    end  pos_1 strand             genes  promoter
0    114    114    114      +        thrA; thrL       NaN
1    147    147    147      +        thrA; thrL       NaN
2   8015   8015   8015      -        yaaJ; talB       NaN
3   8192   8192   8192      +        yaaJ; talB       NaN
4  10564  10564  10564      -  mbiA; satP; yaaW       NaN


Functions for sequence extraction

In [ ]:
def reverse_complement(seq):
    comp = str.maketrans("ACGTN", "TGCAN")
    return seq.translate(comp)[::-1]

def extract_window(genome, center_pos_1based, strand, window_size):
    """
    center_pos_1based: TSS position in 1-based indexing
    strand: '+' or '-'
    window_size: total length of returned sequence
    """
    half = window_size // 2
    center_idx = center_pos_1based - 1  # convert to 0-based

    start = center_idx - half
    end = start + window_size

    if start < 0 or end > len(genome):
        return None

    seq = genome[start:end]

    if strand == "-":
        seq = reverse_complement(seq)

    return seq

In [ ]:
test_seq = extract_window(genome_seq, tss_df.iloc[0]["pos_1"], tss_df.iloc[0]["strand"], 100)
print(test_seq)
print(len(test_seq) if test_seq else None)

TAGCAGCTTCTGAACTGGTTACCTGCCGTGAGTAAATTAAAATTTTATTGACTTAGGTCACTAAATACTTTAACCAATATAGGCATAGCGCACAGACAGA
100


Positive examples

In [ ]:
def build_positive_set(tss_df, genome, window_size):
    rows = []

    for _, row in tss_df.iterrows():
        seq = extract_window(genome, row["pos_1"], row["strand"], window_size)
        if seq is None:
            continue

        rows.append({
            "sequence": seq,
            "label": 1,
            "tss": row["pos_1"],
            "strand": row["strand"],
            "genes": row.get("genes", "")
        })

    return pd.DataFrame(rows)

pos_100 = build_positive_set(tss_df, genome_seq, 100)
pos_200 = build_positive_set(tss_df, genome_seq, 200)
pos_500 = build_positive_set(tss_df, genome_seq, 500)

print(len(pos_100), len(pos_200), len(pos_500))
pos_100.head()

2122 2122 2120


,sequence,label,tss,strand,genes
0,TAGCAGCTTCTGAACTGGTTACCTGCCGTGAGTAAATTAAAATTTT...,1,114,+,thrA; thrL
1,AAATTAAAATTTTATTGACTTAGGTCACTAAATACTTTAACCAATA...,1,147,+,thrA; thrL
2,ATGCTACGCAGAAGTTATCAAGTACCTCGTAGCGTATATACTTCTT...,1,8015,-,yaaJ; talB
3,CCGTCTTGTCGGCGGTTGCGCTGACGTTGCGTCGTGATATCATCAG...,1,8192,+,yaaJ; talB
4,CCCCTCAAAAGATCGTAGACACTGCCCCACTGGCTGATTATTATGC...,1,10564,-,mbiA; satP; yaaW


Negative examples

In [ ]:
tss_positions = sorted(tss_df["pos_1"].unique().tolist())
def is_far_from_tss(pos, tss_positions, min_distance=300):
    # pos is 1-based
    for tss in tss_positions:
        if abs(pos - tss) < min_distance:
            return False
    return True

def build_negative_set(genome, tss_positions, window_size, n_samples, min_distance=300):
    rows = []
    used = set()
    half = window_size // 2

    attempts = 0
    max_attempts = n_samples * 50

    while len(rows) < n_samples and attempts < max_attempts:
        attempts += 1

        pos = random.randint(half + 1, len(genome) - half - 1)

        if pos in used:
            continue

        if not is_far_from_tss(pos, tss_positions, min_distance=min_distance):
            continue

        seq = extract_window(genome, pos, "+", window_size)
        if seq is None:
            continue

        used.add(pos)
        rows.append({
            "sequence": seq,
            "label": 0,
            "tss": pos,
            "strand": "+",
            "genes": ""
        })

    return pd.DataFrame(rows)

neg_100 = build_negative_set(genome_seq, tss_positions, 100, len(pos_100), min_distance=300)
neg_200 = build_negative_set(genome_seq, tss_positions, 200, len(pos_200), min_distance=300)
neg_500 = build_negative_set(genome_seq, tss_positions, 500, len(pos_500), min_distance=500)

print(len(neg_100), len(neg_200), len(neg_500))
neg_100.head()

2122 2122 2120


,sequence,label,tss,strand,genes
0,GTGTTTGCCATCTGTCACGGCCCGCAGTTGCTGATCAGCGCCGATG...,0,3299321,+,
1,GCGTCAGGTGGTGAAAGAACTTGGCATTCCGGTTAACAGCGAACCG...,0,1484913,+,
2,CAAATCGTCCACCTGATCCTGACTGGCTACCACACCGGCTTTGCTG...,0,1966464,+,
3,ACCTGCGGTGGCAAAGCGGTTCGCCAGATCTTGCCCAGTCTGCGAA...,0,700197,+,
4,GACCAGCCCCGGATGGAGCTTGAGTAACTCAACATTAAGTTCTTTG...,0,3401664,+,


Final dataset

In [ ]:
data_100 = pd.concat([pos_100, neg_100], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
data_200 = pd.concat([pos_200, neg_200], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
data_500 = pd.concat([pos_500, neg_500], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
data_100.to_csv("x", index=False)
data_200.to_csv("x", index=False)
data_500.to_csv("x", index=False)

Проверка

In [ ]:
path_100 = "x"
path_200 = "x"
path_500 = "x"

df_100 = pd.read_csv(path_100)
df_200 = pd.read_csv(path_200)
df_500 = pd.read_csv(path_500)

print("100bp:", df_100.shape)
print(df_100["label"].value_counts())
print(df_100.head(), "\n")

print("200bp:", df_200.shape)
print(df_200["label"].value_counts())
print(df_200.head(), "\n")

print("500bp:", df_500.shape)
print(df_500["label"].value_counts())
print(df_500.head())

100bp: (4244, 5)
label
1    2122
0    2122
Name: count, dtype: int64
                                            sequence  label      tss strand  \
0  GCGTTAACCAATCTTCTCTGGATGGGGTATTTAGGGTTACAATATG...      1  4375656      +   
1  TGGTTCTGGTCAGGTGCAGATGCGCCAGGGATTCTTCGCTGTACAG...      0    14712      +   
2  CGCGCGCGAGGCGCTCGCGTGGATTGAAGAAGGGATGGTGATAGCC...      0  2939666      +   
3  GATAGTAAAAGTTTGCAACAAGGGCGAAAGTCAGTACAATCCCCGC...      1   432184      -   
4  TCTTTATATGCTTGATATACTTAAGGTTGTAATAAGCAAAAGAGGA...      0  3121349      +   

             genes  
0  efp; ecnA; epmB  
1              NaN  
2              NaN  
3        tsx; yajI  
4              NaN   

200bp: (4244, 5)
label
1    2122
0    2122
Name: count, dtype: int64
                                            sequence  label      tss strand  \
0  ATTCAAAAGACGCAGAAGTTCATCAGGATCGGTCACAACATCGGCA...      1  4375656      +   
1  TCTTGATATATAACTTTTATTTGCTCAAACGTAAGACGGCATATTT...      0  3798709      +   
2  AACGCCGCCAGACGGGCAA

In [ ]:
df_100["seq_len"] = df_100["sequence"].astype(str).apply(len)
df_200["seq_len"] = df_200["sequence"].astype(str).apply(len)
df_500["seq_len"] = df_500["sequence"].astype(str).apply(len)

print(df_100["seq_len"].value_counts().sort_index())
print(df_200["seq_len"].value_counts().sort_index())
print(df_500["seq_len"].value_counts().sort_index())

seq_len
100    4244
Name: count, dtype: int64
seq_len
200    4244
Name: count, dtype: int64
seq_len
500    4240
Name: count, dtype: int64


Train/Test/Validation

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
path_100 = "x"
path_200 = "x"
path_500 = "x"

df_100 = pd.read_csv(path_100)
df_200 = pd.read_csv(path_200)
df_500 = pd.read_csv(path_500)

def make_splits(df, random_state=42):
    X = df["sequence"].astype(str)
    y = df["label"].astype(int)

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=random_state
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=random_state
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

splits_100 = make_splits(df_100)
splits_200 = make_splits(df_200)
splits_500 = make_splits(df_500)

for name, splits in [("100bp", splits_100), ("200bp", splits_200), ("500bp", splits_500)]:
    X_train, X_val, X_test, y_train, y_val, y_test = splits
    print(f"\n{name}")
    print("Train:", len(X_train), y_train.value_counts().to_dict())
    print("Val:  ", len(X_val), y_val.value_counts().to_dict())
    print("Test: ", len(X_test), y_test.value_counts().to_dict())


100bp
Train: 2970 {0: 1485, 1: 1485}
Val:   637 {1: 319, 0: 318}
Test:  637 {0: 319, 1: 318}

200bp
Train: 2970 {0: 1485, 1: 1485}
Val:   637 {1: 319, 0: 318}
Test:  637 {0: 319, 1: 318}

500bp
Train: 2968 {0: 1484, 1: 1484}
Val:   636 {0: 318, 1: 318}
Test:  636 {1: 318, 0: 318}


# First baseline: k-mer + TF-IDF + Logistic Regression




In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score

In [ ]:
def seq_to_kmers(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq) - k + 1)])

def train_kmer_baseline(X_train, X_val, X_test, y_train, y_val, y_test, k=6):
    X_train_k = X_train.apply(lambda s: seq_to_kmers(s, k))
    X_val_k = X_val.apply(lambda s: seq_to_kmers(s, k))
    X_test_k = X_test.apply(lambda s: seq_to_kmers(s, k))

    vectorizer = TfidfVectorizer()
    X_train_vec = vectorizer.fit_transform(X_train_k)
    X_val_vec = vectorizer.transform(X_val_k)
    X_test_vec = vectorizer.transform(X_test_k)

    clf = LogisticRegression(max_iter=2000, random_state=42)
    clf.fit(X_train_vec, y_train)

    val_pred = clf.predict(X_val_vec)
    test_pred = clf.predict(X_test_vec)
    test_prob = clf.predict_proba(X_test_vec)[:, 1]

    print("Validation Accuracy:", accuracy_score(y_val, val_pred))
    print("Test Accuracy:", accuracy_score(y_test, test_pred))
    print("Test F1:", f1_score(y_test, test_pred))
    print("Test ROC-AUC:", roc_auc_score(y_test, test_prob))
    print("\nClassification Report:\n")
    print(classification_report(y_test, test_pred))

    return clf, vectorizer

In [ ]:
print("===== 100bp =====")
clf_100, vec_100 = train_kmer_baseline(*splits_100, k=6)

print("===== 200bp =====")
clf_200, vec_200 = train_kmer_baseline(*splits_200, k=6)

print("===== 500bp =====")
clf_500, vec_500 = train_kmer_baseline(*splits_500, k=6)

===== 100bp =====
Validation Accuracy: 0.8383045525902669
Test Accuracy: 0.7943485086342229
Test F1: 0.8006088280060882
Test ROC-AUC: 0.8631631868456852

Classification Report:

              precision    recall  f1-score   support

           0       0.82      0.76      0.79       319
           1       0.78      0.83      0.80       318

    accuracy                           0.79       637
   macro avg       0.80      0.79      0.79       637
weighted avg       0.80      0.79      0.79       637

===== 200bp =====
Validation Accuracy: 0.814756671899529
Test Accuracy: 0.7990580847723705
Test F1: 0.8018575851393189
Test ROC-AUC: 0.848021529543976

Classification Report:

              precision    recall  f1-score   support

           0       0.81      0.78      0.80       319
           1       0.79      0.81      0.80       318

    accuracy                           0.80       637
   macro avg       0.80      0.80      0.80       637
weighted avg       0.80      0.80      0.80    

## CNN baseline

In [ ]:
!pip install torch scikit-learn pandas numpy

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
path_100 = "x"
path_200 = "x"
path_500 = "x"

df_100 = pd.read_csv(path_100)
df_200 = pd.read_csv(path_200)
df_500 = pd.read_csv(path_500)

In [ ]:
def make_splits(df, random_state=42):
    X = df["sequence"].astype(str)
    y = df["label"].astype(int)

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=random_state
    )
    return X_train.tolist(), X_val.tolist(), X_test.tolist(), y_train.tolist(), y_val.tolist(), y_test.tolist()

In [ ]:
splits_100 = make_splits(df_100)
splits_200 = make_splits(df_200)
splits_500 = make_splits(df_500)

In [ ]:
mapping = {
    "A": [1,0,0,0],
    "C": [0,1,0,0],
    "G": [0,0,1,0],
    "T": [0,0,0,1],
    "N": [0,0,0,0]
}

def one_hot_encode(seq):
    return np.array([mapping.get(ch, [0,0,0,0]) for ch in seq], dtype=np.float32)

In [ ]:
class PromoterDataset(Dataset):
    def __init__(self, sequences, labels):
        self.X = [one_hot_encode(seq) for seq in sequences]
        self.y = labels

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx], dtype=torch.float32).permute(1, 0)  # [4, L]
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        return x, y

In [ ]:
class CNNPromoter(nn.Module):
    def __init__(self, seq_len):
        super().__init__()
        self.conv1 = nn.Conv1d(4, 64, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(128)
        self.pool2 = nn.MaxPool1d(2)

        self.conv3 = nn.Conv1d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(256)

        self.global_pool = nn.AdaptiveMaxPool1d(1)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = self.pool1(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool2(torch.relu(self.bn2(self.conv2(x))))
        x = torch.relu(self.bn3(self.conv3(x)))
        x = self.global_pool(x).squeeze(-1)
        x = self.dropout(torch.relu(self.fc1(x)))
        x = self.fc2(x).squeeze(-1)
        return x

In [ ]:
def train_model(model, train_loader, val_loader, epochs=10, lr=1e-3):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_f1 = 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        val_metrics = evaluate_model(model, val_loader, return_report=False)
        val_f1 = val_metrics["f1"]

        print(f"Epoch {epoch+1}/{epochs} | Train loss: {np.mean(train_losses):.4f} | Val acc: {val_metrics['accuracy']:.4f} | Val f1: {val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = model.state_dict()

    if best_state is not None:
        model.load_state_dict(best_state)

    return model

In [ ]:
def evaluate_model(model, loader, return_report=True):
    model.eval()
    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= 0.5).astype(int)

            y_prob.extend(probs.tolist())
            y_pred.extend(preds.tolist())
            y_true.extend(yb.numpy().tolist())

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob)
    }

    if return_report:
        print("Accuracy:", metrics["accuracy"])
        print("F1:", metrics["f1"])
        print("ROC-AUC:", metrics["roc_auc"])
        print(classification_report(y_true, y_pred))

    return metrics

In [ ]:
def run_cnn_experiment(splits, batch_size=64, epochs=10, lr=1e-3):
    X_train, X_val, X_test, y_train, y_val, y_test = splits

    train_ds = PromoterDataset(X_train, y_train)
    val_ds = PromoterDataset(X_val, y_val)
    test_ds = PromoterDataset(X_test, y_test)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    seq_len = len(X_train[0])
    model = CNNPromoter(seq_len).to(device)

    model = train_model(model, train_loader, val_loader, epochs=epochs, lr=lr)
    print("\nTest results:")
    test_metrics = evaluate_model(model, test_loader, return_report=True)

    return model, test_metrics

In [ ]:
print("===== CNN 100bp =====")
cnn_100, cnn_metrics_100 = run_cnn_experiment(splits_100, epochs=10)

print("===== CNN 200bp =====")
cnn_200, cnn_metrics_200 = run_cnn_experiment(splits_200, epochs=10)

print("===== CNN 500bp =====")
cnn_500, cnn_metrics_500 = run_cnn_experiment(splits_500, epochs=10)

===== CNN 100bp =====
Epoch 1/10 | Train loss: 0.5517 | Val acc: 0.7582 | Val f1: 0.7381
Epoch 2/10 | Train loss: 0.4376 | Val acc: 0.7975 | Val f1: 0.8245
Epoch 3/10 | Train loss: 0.3260 | Val acc: 0.8195 | Val f1: 0.8311
Epoch 4/10 | Train loss: 0.2330 | Val acc: 0.8116 | Val f1: 0.8276
Epoch 5/10 | Train loss: 0.1370 | Val acc: 0.7457 | Val f1: 0.7944
Epoch 6/10 | Train loss: 0.0932 | Val acc: 0.8163 | Val f1: 0.8091
Epoch 7/10 | Train loss: 0.0564 | Val acc: 0.7991 | Val f1: 0.7698
Epoch 8/10 | Train loss: 0.0210 | Val acc: 0.8399 | Val f1: 0.8440
Epoch 9/10 | Train loss: 0.0114 | Val acc: 0.8195 | Val f1: 0.8142
Epoch 10/10 | Train loss: 0.0051 | Val acc: 0.8336 | Val f1: 0.8404

Test results:
Accuracy: 0.8288854003139717
F1: 0.8350983358547656
ROC-AUC: 0.8985627254983143
              precision    recall  f1-score   support

         0.0       0.86      0.79      0.82       319
         1.0       0.80      0.87      0.84       318

    accuracy                           0.83     